## LAB: TEXT MINING - REGULAR EXPRESSIONS

In [7]:
import re
import pandas as pd

with open("/Users/christinelezama/Downloads/fraudulent_emails.txt",
          "r", encoding="latin-1") as f:
    data = f.read()

lines = data.splitlines()

print(lines[:10])

['From r  Wed Oct 30 21:41:56 2002', 'Return-Path: <james_ngola2002@maktoob.com>', 'X-Sieve: cmu-sieve 2.0', 'Return-Path: <james_ngola2002@maktoob.com>', 'Message-Id: <200210310241.g9V2fNm6028281@cs.CU>', 'From: "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>', 'Reply-To: james_ngola2002@maktoob.com', 'To: webmaster@aclweb.org', 'Date: Thu, 31 Oct 2002 02:38:20 +0000', 'Subject: URGENT BUSINESS ASSISTANCE AND PARTNERSHIP']


In [11]:
from_lines = []

for line in lines:
    if line.startswith("From:"):
        from_lines.append(line)

for line in from_lines[:20]:
    print(line)

From: "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>
From: "Mr. Ben Suleman" <bensul2004nng@spinfinder.com>
From: "PRINCE OBONG ELEME" <obong_715@epatra.com>
From: "PRINCE OBONG ELEME" <obong_715@epatra.com>
From: "Maryam Abacha" <m_abacha03@www.com>
From: Kuta David <davidkuta@postmark.net>
From: "Barrister tunde dosumu" <tunde_dosumu@lycos.com>
From: "William Drallo" <william2244drallo@maktoob.com>
From: "MR USMAN ABDUL" <abdul_817@rediffmail.com>
From: "Tunde  Dosumu" <barrister_td@lycos.com>
From: MR TEMI JOHNSON <temijohnson2@rediffmail.com>
From: "Dr.Sam jordan" <sjordan@diplomats.com>
From: p_brown2@lawyer.com
From: mic_k1@post.com
From: "COL. MICHAEL BUNDU" <mikebunduu1@rediffmail.com>
From: "MRS MARIAM ABACHA" <elixwilliam@usa.com>
From: " DR. ANAYO AWKA " <anayoawka@hotmail.com>
From: " DR. ANAYO AWKA " <anayoawka@hotmail.com>
From: "Victor Aloma" <victorloma@netscape.net>
From: "Victor Aloma" <victorloma@netscape.net>


In [15]:
pat = r'^From:\s*"?\s*Dr\.?\b\s*'
regexp = re.compile(pat, re.IGNORECASE)

doctor_lines = []

for line in lines:
    if regexp.search(line):
        doctor_lines.append(line)

for line in doctor_lines[:10]:
    print(line)
    

From: "Dr.Sam jordan" <sjordan@diplomats.com>
From: " DR. ANAYO AWKA " <anayoawka@hotmail.com>
FROM: DR. ANAYO AWKA BANK MANAGER 
From: " DR. ANAYO AWKA " <anayoawka@hotmail.com>
FROM: DR. ANAYO AWKA BANK MANAGER 
From: "DR. JAMES  ALBERT" <jamesalbert0@eircom.net>
FROM: DR, ADAMS OBOZUWA. 
FROM: DR, IDRIS BELLO. 
FROM: DR, IDRIS BELLO. 
FROM: DR, IDRIS BELLO. 


In [19]:
pat = r'^From:\s*"?\s*(Dr\b\.?[^"<>\r\n]*)"?\s*<([\w.+-]+@([\w.-]+\.(?:com|net)))>\s*$'
regexp = re.compile(pat, re.IGNORECASE)

records = []

for line_number, line in enumerate(lines):
    match = regexp.search(line)

    if match:
        records.append([
            line_number,
            match.group(1).strip(),
            match.group(2),
            match.group(3)
        ])

df = pd.DataFrame(
    records,
    columns=["Line Number", "Name", "E-mail Address", "E-mail Domain"]
)

print("Records matched:", len(df))
df.head(21)

Records matched: 144


,Line Number,Name,E-mail Address,E-mail Domain
0,746,Dr.Sam jordan,sjordan@diplomats.com,diplomats.com
1,1117,DR. ANAYO AWKA,anayoawka@hotmail.com,hotmail.com
2,1155,DR. ANAYO AWKA,anayoawka@hotmail.com,hotmail.com
3,2067,DR. JAMES ALBERT,jamesalbert0@eircom.net,eircom.net
4,3837,DR. SANNI SABO,davidmensoh@mail.com,mail.com
5,3919,DR. SANNI SABO,davidmensoh@mail.com,mail.com
6,4860,DR. SANNI SABO,davidmensoh@mail.com,mail.com
7,4919,Dr Luisa Pimental Estrada,luisaestrada2002@luisapestrada.zzn.com,luisapestrada.zzn.com
8,5624,DR. BELLO MUSA,dr_bellomusa01@omaninfo.com,omaninfo.com
9,7289,Dr mariam abacha,marabac011@hotmail.com,hotmail.com


In [25]:
date_pat = r'^Date:\s*(\w{3}),?\s+(\d{1,2}\s+\w{3}\s+\d{4})\s+(\d{2}:\d{2}:\d{2})'
date_regexp = re.compile(date_pat, re.IGNORECASE)

example = "Date: Fri, 08 Nov 2002 04:15:33"
match = date_regexp.search(example)

print("Day:", match.group(1))
print("Date:", match.group(2))
print("Time:", match.group(3))

Day: Fri
Date: 08 Nov 2002
Time: 04:15:33


In [27]:
print("Records matched:", len(df))

Records matched: 144


In [30]:
date_pat = r'^Date:\s*(\w{3}),?\s+(\d{1,2}\s+\w{3}\s+\d{2}(?:\d{2})?)\s+(\d{2}:\d{2}:\d{2})'
date_regexp = re.compile(date_pat, re.IGNORECASE)

timestamps = []

for line_number in df["Line Number"]:
    start = line_number
    end = line_number

    # Find the beginning of this email's header
    while start > 0:
        previous = lines[start - 1]

        if previous.strip() == "" or previous.startswith("From "):
            break

        start -= 1

    # Find the end of this email's header
    while end < len(lines):
        if lines[end].strip() == "":
            break

        end += 1

    timestamp = [None, None, None]

    for line in lines[start:end]:
        match = date_regexp.search(line)

        if match:
            timestamp = [
                match.group(1),
                match.group(2),
                match.group(3)
            ]
            break

    timestamps.append(timestamp)

df[["Day", "Date", "Time"]] = timestamps

print("Missing timestamps:", df["Date"].isna().sum())
df.head(21)

Missing timestamps: 5


,Line Number,Name,E-mail Address,E-mail Domain,Day,Date,Time
0,746,Dr.Sam jordan,sjordan@diplomats.com,diplomats.com,Fri,08 Nov 2002,04:15:33
1,1117,DR. ANAYO AWKA,anayoawka@hotmail.com,hotmail.com,Fri,15 Nov 2002,00:40:13
2,1155,DR. ANAYO AWKA,anayoawka@hotmail.com,hotmail.com,Fri,15 Nov 2002,01:18:50
3,2067,DR. JAMES ALBERT,jamesalbert0@eircom.net,eircom.net,Tue,26 Nov 2002,04:34:05
4,3837,DR. SANNI SABO,davidmensoh@mail.com,mail.com,Fri,1 Jan 1999,22:08:55
5,3919,DR. SANNI SABO,davidmensoh@mail.com,mail.com,Sat,2 Jan 1999,01:43:13
6,4860,DR. SANNI SABO,davidmensoh@mail.com,mail.com,Fri,1 Jan 1999,11:44:49
7,4919,Dr Luisa Pimental Estrada,luisaestrada2002@luisapestrada.zzn.com,luisapestrada.zzn.com,Sat,18 Jan 2003,15:49:19
8,5624,DR. BELLO MUSA,dr_bellomusa01@omaninfo.com,omaninfo.com,Thu,23 Jan 2003,04:16:53
9,7289,Dr mariam abacha,marabac011@hotmail.com,hotmail.com,Fri,7 Feb 2003,17:38:37
